# 02 · 模型训练（Colab）—— v2：DNS Challenge 真实噪声

训练 `crn-nano` / `crn-lite` / `crn-large` 三档模型，跟 v1 用完全相同的模型代码、
超参、训练流程——**唯一变量是数据源**（噪声/RIR 换成真实 DNS 数据），
这样最终对比才能把"结构变了"和"数据变了"这两件事分开。

训练结果保存到 `DRIVE_ROOT/checkpoints/`（v2 专属路径，不会覆盖 v1 的）。

In [ ]:
# ── 挂载 Google Drive ───────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
#  配置 —— 所有路径与选择都集中在这一个 cell，别处不要再写死路径
#  这是 v2（DNS Challenge 真实噪声）。v1（THCHS-30+MUSAN）在
#  notebooks/v1_thchs30_musan/，两版数据/checkpoint 各自独立，不会互相覆盖。
# ═══════════════════════════════════════════════════════════════════════

# rtse-colab.zip 所在目录——**v1/v2 共用同一份代码包**，不需要重复上传。
# 如果你还没跑过 v1，这里跟下面 DRIVE_ROOT 的上一级目录填一样的就行。
CODE_ROOT = '/content/drive/MyDrive/Audio AI/RTSE'

# 本版本专属的数据/checkpoint/测试集根目录，嵌套在 CODE_ROOT 下面，
# 与 v1 直接写在 CODE_ROOT 下的 manifest.json/checkpoints/models/testset 互不冲突。
DRIVE_ROOT = '/content/drive/MyDrive/Audio AI/RTSE/v2_dns_real_noise'

# Colab 本地盘。**临时**，会话结束即消失，但读写比 Drive 快得多。
WORK_ROOT = '/content/rtse_work'

# ── 原始语料怎么放？────────────────────────────────────────────────────
#
#   'hybrid' —— **推荐**。压缩包缓存在 Drive，每次会话解压到本地盘。
#               一次下载永久有效；训练读取是本地盘全速；
#               每个新会话只需几分钟解压。
#
#   'local'  —— 全部放本地盘，压缩包用完即删。
#               不占 Drive；代价是**每次新会话都要重下**。
#
#   'drive'  —— 全部放 Drive。THCHS-30 一万多个小文件在 FUSE 挂载上解压很慢，
#               DNS 噪声/IR 是几个大文件（不是海量小文件）不受这个问题影响，
#               但仍不推荐——训练时随机读取 Drive 比本地盘慢。
DATA_MODE = 'hybrid'

# ── 快速验证模式 ────────────────────────────────────────────────────────
# True  = 只下载 337 MB 的小语料（LibriSpeech dev-clean，英文），不碰 DNS 数据，
#         约 15 分钟验证「数据→训练→导出→回传」整条链路。
#         **只用来验证流程，不要用它的结果做最终指标**。
# False = 完整流程：THCHS-30 中文语音 + DNS Challenge 真实噪声/RIR
QUICK_TEST = False

# ═══════════════════════════════════════════════════════════════════════

import os, sys, json, shutil, subprocess, time
from pathlib import Path
from shlex import quote as shq          # 路径里有空格时，所有 shell 命令都要靠它

assert DATA_MODE in ('hybrid', 'local', 'drive'), f'DATA_MODE 只能是 hybrid/local/drive'

CODE = CODE_ROOT
DRIVE = DRIVE_ROOT
WORK = WORK_ROOT

# 压缩包放哪 / 解压到哪 —— 三种模式的唯一区别就在这两行
ARCHIVE_DIR = f'{WORK}/archives' if DATA_MODE == 'local' else f'{DRIVE}/archives'
DATA = f'{DRIVE}/rawdata' if DATA_MODE == 'drive' else f'{WORK}/data'
KEEP_ARCHIVE = DATA_MODE != 'local'    # local 模式解压后删包省空间，其余保留以便复用

# Drive 侧的产物目录（**这些永远在 Drive 上**，训练结果不能放临时盘）
CKPT_DIR = f'{DRIVE}/checkpoints'   # 训练断点，每个 epoch 保存
MODEL_DIR = f'{DRIVE}/models'       # 导出的 ONNX
TESTSET_DIR = f'{DRIVE}/testset'    # 固定测试集
LOG_DIR = f'{DRIVE}/logs'

assert os.path.isdir(CODE), (
    f'Drive 上找不到 {CODE}\n'
    '检查两件事：① Drive 已挂载成功；② CODE_ROOT 与你实际存放 rtse-colab.zip 的目录一致。'
)
for d in [WORK, ARCHIVE_DIR, DATA, CKPT_DIR, MODEL_DIR, TESTSET_DIR, LOG_DIR]:
    os.makedirs(d, exist_ok=True)

print('目录布局（v2 · DNS Challenge 真实噪声）')
print('─' * 74)
print(f'  代码包(v1/v2 共用)  {CODE}/rtse-colab.zip')
print(f'  压缩包缓存          {ARCHIVE_DIR}')
print(f'  语料解压目标        {DATA}')
print(f'  数据清单            {DRIVE}/manifest.json')
print(f'  固定测试集          {TESTSET_DIR}')
print(f'  训练断点  ★         {CKPT_DIR}/<模型名>/{{last,best}}.pt')
print(f'  导出模型  ★         {MODEL_DIR}/<模型名>.onnx')
print('─' * 74)
print(f'  数据模式  {DATA_MODE}')
print(f'  语料      {"快速验证(小语料/英文)" if QUICK_TEST else "完整流程(THCHS-30 中文语音 + DNS 真实噪声/RIR)"}')
print()

!df -h /content | tail -1
!df -h /content/drive 2>/dev/null | tail -1

In [ ]:
# ── 安装项目代码 ────────────────────────────────────────────────────────
# rtse-colab.zip 由本地 `uv run python scripts/pack_for_colab.py` 生成，
# 需要手动上传到 CODE_ROOT 指向的目录（v1/v2 共用同一份，不用重新上传）。
ZIP = f'{CODE}/rtse-colab.zip'
assert os.path.exists(ZIP), (
    f'找不到 {ZIP}\n'
    '请先在本地执行 `uv run python scripts/pack_for_colab.py`，'
    f'再把 dist/rtse-colab.zip 上传到 Drive 的 {CODE} 下。\n'
    f'该目录下现有：{sorted(os.listdir(CODE))[:12]}'
)

SRC = f'{WORK}/rtse-src'
shutil.rmtree(SRC, ignore_errors=True)
os.makedirs(SRC, exist_ok=True)
!unzip -q -o {shq(ZIP)} -d {shq(SRC)}

# 只装项目需要而 Colab 没有预装的几个包。
# 不用 `pip install -e .`：那会去解析 pyproject 里锁定的 torch CPU 索引，
# 把 Colab 自带的 GPU 版 torch 覆盖掉 —— 训练会瞬间慢几十倍。
!pip install -q soxr pystoi jiwer webrtcvad-wheels pesq onnx onnxruntime 2>&1 | tail -2

sys.path.insert(0, f'{SRC}/src')
import rtse
print('rtse', rtse.__version__, '| SR', rtse.SAMPLE_RATE, '| n_fft', rtse.N_FFT, '| hop', rtse.HOP_LENGTH)

import torch
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
# ── 自检：Colab 侧与本地必须是同一条信号链路 ───────────────────────────
# 这一步不能跳。如果 Colab 上的 STFT 与本地哪怕差一点，
# 训练出来的模型拿回本地就会掉点，而且极难定位（两边单独看都"没问题"）。
import numpy as np, torch
from rtse.audio.stft import stft, istft, check_cola, magnitude_db
from rtse.data.dataset import stft_torch, istft_torch

x = np.random.default_rng(0).standard_normal(16000)
print('COLA 偏差          :', f'{check_cola():.2e}')
print('numpy 完美重构      :', f'{np.max(np.abs(istft(stft(x), length=x.size) - x)):.2e}')

xt = torch.from_numpy(x).float().unsqueeze(0)
ref, got = stft(x), stft_torch(xt)
assert ref.shape[0] == got.shape[2], f'帧数不一致 {ref.shape[0]} vs {got.shape[2]}'
gc = got[0,0].numpy() + 1j*got[0,1].numpy()
print('torch/numpy STFT   :', f'{np.max(np.abs(gc - ref)) / np.max(np.abs(ref)):.2e} (相对)')
print('torch 往返重构      :', f'{(istft_torch(stft_torch(xt), length=16000) - xt).abs().max().item():.2e}')

t = np.arange(16000)/16000
db = magnitude_db(stft(np.sin(2*np.pi*1000*t))).max()
print('dBFS 标定(满幅正弦) :', f'{db:.3f} dB  (应为 0.000)')
assert abs(db) < 0.05, 'dBFS 标定不对，检查代码包是否为最新'
print('\n✅ Colab 与本地是同一条链路。')

## 1. 构建数据集

跟 v1 一样：Colab 每次连接都是全新虚拟机，本地盘要重新解压。这里复用 01 里同一套
`fetch_openslr` / `fetch_dns_blob`，已经下载过的会秒过。

In [ ]:
mf = Path(f'{DRIVE}/manifest.json')
assert mf.exists(), f'找不到 {mf}，先跑 01_data_prep.ipynb（v2 版）'
manifest = json.loads(mf.read_text(encoding='utf-8'))
print(f'清单来源: {manifest["data_dir"]}   version={manifest.get("version")}   quick_test={manifest["quick_test"]}')

In [ ]:
QUICK_TEST = manifest['quick_test']  # 以清单记录的实际取值为准

if QUICK_TEST:
    SPEECH_DATASETS = [
        ('librispeech', 12, 'dev-clean.tar.gz', 0.34, 'LibriSpeech'),
    ]
    DNS_NOISE_SHARDS, DNS_IR_SHARDS = [], []   # 快速模式不碰 DNS，走合成噪声/RIR
else:
    SPEECH_DATASETS = [
        ('thchs30', 18, 'data_thchs30.tgz', 6.4, 'data_thchs30'),
    ]
    # 2 个真实噪声分片（AudioSet 来源的日常环境声 + Freesound 来源的标注音效），
    # 覆盖面已经比 MUSAN 的 noise 子集广很多；1 个房间冲激响应分片（真实测量+仿真混合）。
    # "有代表性的小规模子集"：这 3 个分片总计约 20GB，相对完整 DNS5 语料（噪声+IR 约 64GB，
    # 加上多语言干净语音后总计约 890GB）是一个刻意选小的子集，不追求覆盖全部分片。
    # 单个分片体积未逐一实测（本机磁盘装不下拿来验证），下方按官方文档给出的
    # "全部噪声分片共 ~39GB / 9 个分片"估算均摊值，供磁盘预算做粗略预检，
    # 实际大小以下载时 wget 显示的为准。
    DNS_NOISE_SHARDS = [
        ('dns_noise_audioset', 'noise_fullband/datasets_fullband.noise_fullband.audioset_000.tar.bz2', 4.3),
        ('dns_noise_freesound', 'noise_fullband/datasets_fullband.noise_fullband.freesound_000.tar.bz2', 4.3),
    ]
    DNS_IR_SHARDS = [
        ('dns_ir', 'datasets_fullband.impulse_responses_000.tar.bz2', 5.9),
    ]

MIRRORS = [
    'https://www.openslr.org/resources',
    'https://openslr.elda.org/resources',          # 欧洲
    'https://openslr.magicdatatech.com/resources', # 中国
]

def fetch_openslr(name, slr, fname, expect_dir):
    """下载 → 解压 → 校验（openSLR 镜像，.tgz/.zip）。跟 v1 是同一份逻辑。"""
    dl_mark = f'{ARCHIVE_DIR}/.{name}.downloaded'
    ex_mark = f'{DATA}/.{name}.extracted'
    archive = f'{ARCHIVE_DIR}/{fname}'

    if os.path.exists(ex_mark) and os.path.isdir(f'{DATA}/{expect_dir}'):
        print(f'[skip  ] {name} 已解压'); return True

    if os.path.exists(dl_mark) and os.path.exists(archive):
        print(f'[cached] {name} 压缩包已在 Drive ({os.path.getsize(archive)/1e9:.2f} GB)，跳过下载')
    else:
        ok = False
        for base in MIRRORS:
            url = f'{base}/{slr}/{fname}'
            print(f'[get   ] {name} ← {url}')
            rc = os.system(f'wget -q --show-progress -c -T 30 -O {shq(archive)} {shq(url)}')
            if rc == 0 and os.path.exists(archive) and os.path.getsize(archive) > 1e6:
                ok = True
                break
            print('[retry ] 该镜像失败，换下一个')
        if not ok:
            print(f'[FAIL  ] {name} 三个镜像都下不下来。'
                  f'去 https://www.openslr.org/{slr}/ 确认文件名是否变了。')
            return False
        Path(dl_mark).touch()

    print(f'[unpack] {name}  ({os.path.getsize(archive)/1e9:.2f} GB) → {DATA}')
    t = time.time()
    if fname.endswith('.zip'):
        rc = os.system(f'unzip -q -o {shq(archive)} -d {shq(DATA)}')
    else:
        rc = os.system(f'tar -xzf {shq(archive)} -C {shq(DATA)}')

    if not os.path.isdir(f'{DATA}/{expect_dir}'):
        print(f'[FAIL  ] 解压后没有找到 {DATA}/{expect_dir}（rc={rc}）')
        print(f'         压缩包可能不完整，删掉 {archive} 和 {dl_mark} 后重跑本 cell')
        return False

    if not KEEP_ARCHIVE:
        os.remove(archive)
        Path(dl_mark).unlink(missing_ok=True)
    Path(ex_mark).touch()
    print(f'[done  ] {name}   解压耗时 {time.time()-t:.0f} 秒')
    return True

DNS_BASE = 'https://dnschallengepublic.blob.core.windows.net/dns5archive/V5_training_dataset'

def fetch_dns_blob(name, blob_path, expect_min_wavs=1):
    """下载 → 解压 DNS Challenge 的一个 .tar.bz2 分片。

    跟 fetch_openslr 用同一套"下载标记 + 解压标记"续传逻辑，但解压用 bz2
    （`tar -xjf`），而且**不假设解压后的内部目录名**——DNS Challenge 官方仓库
    没有在文档里给出每个分片解压后的确切子目录结构，本机磁盘装不下几 GB 的
    分片来提前验证（见 docs/ENVIRONMENT.md，C: 盘预算 <5GB），所以校验方式
    改成"解压后目录树里递归扫到的 wav 数量"，而不是断言一个具体子目录名——
    这样即使 DNS 内部打包结构和预期不同，只要文件确实解出来了就能识别成功，
    下游 scan() 本来就是递归扫描，不关心具体嵌套了几层。
    """
    dl_mark = f'{ARCHIVE_DIR}/.{name}.downloaded'
    ex_mark = f'{DATA}/.{name}.extracted'
    fname = blob_path.rsplit('/', 1)[-1]
    archive = f'{ARCHIVE_DIR}/{fname}'
    out_dir = f'{DATA}/{name}'
    os.makedirs(out_dir, exist_ok=True)

    def count_wavs():
        r = subprocess.run(f'find {shq(out_dir)} -name "*.wav" | wc -l',
                           shell=True, capture_output=True, text=True)
        return int((r.stdout or '0').strip() or 0)

    if os.path.exists(ex_mark):
        n = count_wavs()
        if n >= expect_min_wavs:
            print(f'[skip  ] {name} 已解压（{n} 个 wav）'); return True

    if os.path.exists(dl_mark) and os.path.exists(archive):
        print(f'[cached] {name} 压缩包已在 Drive ({os.path.getsize(archive)/1e9:.2f} GB)，跳过下载')
    else:
        url = f'{DNS_BASE}/{blob_path}'
        print(f'[get   ] {name} ← {url}')
        rc = os.system(f'wget -q --show-progress -c -T 60 -O {shq(archive)} {shq(url)}')
        if rc != 0 or not os.path.exists(archive) or os.path.getsize(archive) < 1e6:
            print(f'[FAIL  ] {name} 下载失败。'
                  f'去 https://github.com/microsoft/DNS-Challenge 确认 blob 路径是否变了。')
            return False
        Path(dl_mark).touch()

    print(f'[unpack] {name}  ({os.path.getsize(archive)/1e9:.2f} GB) → {out_dir}')
    t = time.time()
    rc = os.system(f'tar -xjf {shq(archive)} -C {shq(out_dir)}')
    n = count_wavs()
    if n < expect_min_wavs:
        print(f'[FAIL  ] 解压后只找到 {n} 个 wav 文件（rc={rc}），压缩包可能不完整')
        print(f'         删掉 {archive} 和 {dl_mark} 后重跑本 cell')
        return False

    if not KEEP_ARCHIVE:
        os.remove(archive)
        Path(dl_mark).unlink(missing_ok=True)
    Path(ex_mark).touch()
    print(f'[done  ] {name}   {n} 个 wav   解压耗时 {time.time()-t:.0f} 秒')
    return True

results = {n: fetch_openslr(n, s, f, d) for n, s, f, _, d in SPEECH_DATASETS}
for n, blob, _gb in DNS_NOISE_SHARDS:
    results[n] = fetch_dns_blob(n, blob, expect_min_wavs=100)
for n, blob, _gb in DNS_IR_SHARDS:
    results[n] = fetch_dns_blob(n, blob, expect_min_wavs=50)
assert all(results.values()), '有语料没能自动补齐，需要重新下载，看上面的 FAIL 信息'
print('语料就绪。')

In [ ]:
from torch.utils.data import DataLoader
from rtse.data.dataset import OnlineMixDataset, MixConfig
from rtse.metrics.intrusive import si_sdr

# OnlineMixDataset 只认文件路径列表，不关心噪声/RIR 是合成的还是真实录制的——
# 跟 v1 相比这里的代码完全没变，变化全部发生在 manifest.json 里的路径来源。
MIX = MixConfig(segment_seconds=4.0, snr_range=(-5.0, 20.0), reverb_prob=0.5)

train_ds = OnlineMixDataset(manifest['speech']['train'], manifest['noise_train'],
                            manifest['rir_train'], cfg=MIX, length=20000, seed=0)
val_ds = OnlineMixDataset(manifest['speech']['val'], manifest['noise_test'],
                          manifest['rir_test'], cfg=MIX, length=800, seed=999)

train_dl = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=2,
                      pin_memory=True, drop_last=True, persistent_workers=True)
val_dl = DataLoader(val_ds, batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

nb_, cl_ = next(iter(train_dl))
print('batch', tuple(nb_.shape), '| 输入 SI-SDR 抽样:',
      [round(si_sdr(cl_[i].numpy(), nb_[i].numpy()), 1) for i in range(4)], 'dB')

## 2. 训练

跟 v1 一样的建议：先用 `EPOCHS=5` 跑一轮确认没问题，再改回 60。Colab Pro 的 T4
比免费版排队更少、单会话时限更长，但显卡本身规格相同，训练速度预期跟 v1 接近。

In [ ]:
from rtse.models import build_model
from rtse.train import Trainer, TrainConfig

MODELS = ['crn-nano', 'crn-lite']      # 想跑大模型就加上 'crn-large'
EPOCHS = 60

for name in MODELS:
    out_dir = f'{CKPT_DIR}/{name}'
    os.makedirs(out_dir, exist_ok=True)
    cfg = TrainConfig(model=name, epochs=EPOCHS, batch_size=16, lr=3e-4,
                      out_dir=out_dir, num_workers=2, log_every=100)
    model = build_model(name)
    print(f'\n{"="*70}\n{name}   参数量 {model.count_params():,}   → {out_dir}\n{"="*70}')

    tr = Trainer(model, train_dl, val_dl, cfg)
    last = Path(out_dir) / 'last.pt'
    if last.exists():
        tr.load(last)
    if tr.epoch >= EPOCHS:
        print(f'{name} 已完成（epoch {tr.epoch}），跳过'); continue
    tr.fit()

print('\n训练产物（在 Drive 上，会话断了也在）：')
!ls -lh {shq(CKPT_DIR)}/*/

## 3. 训练曲线

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for name in MODELS:
    h = Path(f'{CKPT_DIR}/{name}/history.json')
    if not h.exists(): continue
    hist = json.loads(h.read_text())
    ep = [r['epoch'] for r in hist]
    axes[0].plot(ep, [r.get('loss') for r in hist], label=name)
    axes[1].plot(ep, [r.get('si_sdr') for r in hist], label=f'{name} train')
    if 'val_si_sdr' in hist[0]:
        axes[1].plot(ep, [r.get('val_si_sdr') for r in hist], '--', label=f'{name} val')
    axes[2].plot(ep, [r.get('spec') for r in hist], label=name)

for ax, t in zip(axes, ['总损失', 'SI-SDR (dB)', '压缩谱损失']):
    ax.set_title(t); ax.set_xlabel('epoch'); ax.grid(alpha=.3); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()